# పాఠం 12 - ఏజెంట్ స్క్రాచ్ప్యాడ్‌తో చాట్ చరిత్ర తగ్గింపు

ఈ నోట్బుక్ మైక్రోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్ ఉపయోగించి పొడవైన సంభాషణలలో సందర్భాన్ని ఎలా నిర్వహించాలో చూపిస్తుంది. సంభాషణలు పెరిగేకొద్దీ టోకెన్ సంఖ్య పెరుగుతుంది — చివరకు మోడల్ యొక్క సాందర్భిక విండో దాటుతోంది. దీని పరిష్కారంగా మనం **సందర్భ సారాంశ నమూనా** మరియు **ఏజెంట్ స్క్రాచ్ప్యాడ్** ని పరిచయం చేస్తాము, ఇది ఎప్పటికప్పుడు నిలిచిపోయే మెమరీ కోసం ఉపయోగపడుతుంది.

## మీరు నేర్చుకోబోయే విషయాలు:
1. **సందర్భ నిర్వహణ ఎందుకు ముఖ్యమో**: టోకెన్ పరిమితులు మరియు సాందర్భిక విండోలను అర్థం చేసుకోవడం
2. **సందర్భ అవగాహన ఉన్న ఏజెంట్లు**: తమ స్వంత సంభాషణ సందర్భాన్ని నిర్వహించే ఏజెంట్లను రూపొందించడం
3. **సందర్భ సారాంశ నమూనా**: సంభాషణ చరిత్రను సంక్షిప్తం చేయడానికి టూల్స్ ఉపయోగించడం
4. **ఏజెంట్ స్క్రాచ్ప్యాడ్**: సందర్భ తగ్గింపు సమయంలో నిలిచిపోయే మెమరీ

## ముందస్తు అవసరాలు:
- Azure OpenAI సెట్‌అప్‌తో పర్యావరణ చరాలు కన్ఫిగర్ చేయబడినవి
- మునుపటి పాఠాల నుండి ప్రాథమిక ఏజెంట్ భావాలను అర్థం చేసుకోవడం


## సెటప్


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## సందర్భ నిర్వహణ ముఖ్యమైంది ఎందుకు

ప్రతి LLM కు ఒక పరిమిత **సందర్భ విండో** ఉంటుంది — ఒకే అభ్యర్థనలో అది ప్రాసెస్ చేయగల టోకెన్ల గరిష్ట సంఖ్య. బహుళపర్యాయ సంభాషణ అభివృద్ధి చెందడంతో:

- ప్రతి యూజర్ సందేశం మరియు అసిస్టెంట్ జవాబుతో **టోకెన్ లెక్క రేఖీయంగా పెరుగుతుంది**.
- **ప్రాంప్ట్ టోకెన్లు ఖర్చు మీద ఆధిపత్యం వహిస్తాయి** ఎందుకంటే మొత్తం చరిత్ర ప్రతి తరం తిరిగి పంపబడుతుంది.
- చివరికి సంభాషణ **సందర్భ విండోను మించినప్పుడు** మోడల్ లేదా కట్ చేస్తుంది లేదా లోపం చూపిస్తుంది.

### సందర్భం నిర్వహణకు వ్యూహాలు

| వ్యూహం | అది ఎలా పనిచేస్తుంది | వ్యతిరేకఫలం |
|---|---|---|
| **కట్** | పాత సందేశాలను వదిలివేయడం | ప్రారంభ సందర్భాన్ని కోల్పోతుంది |
| **సారాంశం** | పాత సందేశాలను సారాంశంగా సంక్షిప్తం చేయడం | కొంత వివరాలు కోల్పోతాయ్, కానీ ముఖ్యాంశాలు నిల్వ ఉంటాయి |
| **స్రాచ్‌ప్యాడ్ / బాహ్య మెమరీ** | సంభాషణ వెలుపల కీలక వాస్తవాలను నిల్వ చేయడం | సాధన పిలుపులు అవసరం, కానీ ఏ తగ్గింపును తట్టుకుంటుంది |

ఈ నోట్‌బుక్‌లో మేము **సారాంశం** ని **స్రాచ్‌ప్యాడ్ సాధనంతో** కలిపి ఉపయోగిస్తాము కావున ఏజెంట్ సంభాషణ చరిత్ర సంక్షిప్తం అయినప్పటికీ నిరంతరతని నిలుపుకోవచ్చు.


## సందర్భాన్ని గుర్తించే ఏజెంట్ ని సృష్టించడం


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## దీర్ఘ సంభాషణను అనుకరిస్తోంది

సందర్భం ఎలా చేరుకుంటుందో చూడడానికి బహుళ-తిరుగుబాటు సంభాషణను మనం పరిగణిస్తాం. ఏజెంట్ కీలక వివరాలు (ఆసక్తులు, బడ్జెట్, ప్రయాణ తేదీలను) తిరుగుబాటు ల ద్వారా నిలుపుకోవాలి మరియు కొనసాగింపును ప్రదర్శించాలి.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

ఏజెంట్ ఎలా ముందటి సంభాషణల నుండి సందర్భాన్ని కలిగి ఉంచుతుందో గమనించండి — అది జపాన్, సుషి, దేవాలయాలు, ఫొటోగ్రఫీ, $3000 బడ్జెట్, ఒక్కొక్కటి ప్రయాణం మరియు ఏప్రిల్ కాలం గురించి తెలుసుకుంటుంది. చిన్న సంభాషణలో ఇది బాగా పనిచేస్తుంది, కానీ సంభాషణ పెరిగే కొద్దీ పూర్తి చరిత్రను మళ్లీ పంపడం ఖర్చుతో ఉంటుంది.

సందర్భ సేకరణని చూడటానికి మరిన్ని సంభాషణలతో కొనసాగుదాం:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Context Summarization Pattern

సంభాషణ పెరిగే కొద్దీ, క్లిష్టమైన సందర్భాన్ని సంక్షిప్త రూపంలో మార్చడానికి మనం **సారాంశ టూల్** ను ఉపయోగించవచ్చు. ఏజెంట్ ఈ టూల్ ను ఆహ్వానించి ముఖ్యమైన ఇష్టాలను నమోదు చేస్తుంది, తద్వారా పాత మెసెజ్లు కోల్పోయినపటికీ కూడా ముఖ్యమైన సమాచారం నిల్వ ఉంటుంది.

ఈ నమూనా మరింత విజ్ఞానవంతమైన చరిత్ర తగ్గింపుకు బ్లాక్‌లుగా పనిచేస్తుంది:
1. ఏజెంట్ సంభాషణ నుండి ముఖ్యమైన విషయాలను గుర్తిస్తుంది
2. వాటిని నిల్వ చేయడానికి సారాంశ టూల్ ను పిలుస్తుంది
3. సారాంశం ముఖ్యమైన విషయాలను రికార్డ్ చేయటంతో పాత మెసెజ్లు సురక్షితంగా తొలగించవచ్చు

క్రింద, ఏజెంట్ నేర్చుకున్న వాటి సంక్షిప్త సారాంశాన్ని నమోదు చేయడానికి `summarize_preferences` అనే టూల్ ను నిర్వచించాము.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## సారాంశం

ఈ పాఠంలో మీరు Microsoft Agent Framework ఉపయోగించి దీర్ఘకాల agent సంభాషణలలో context ను ఎలా నిర్వహించాలో నేర్చుకున్నారు:

### ప్రధాన భావనలు
- **Context విండోస్ పరిమితంగా ఉంటాయి** — సంభాషణ చరిత్రలో ప్రతి టోకెన్ కు ఖర్చు ఉంటది మరియు అది పరిమితికి లెక్కబడుతుంది.
- **సారాంశ అంశాల సాధనాలు** agent కు కూడుకున్న context ను సంక్షిప్త సారాంశాలలో మార్చే అవకాశం ఇస్తాయి, టోకెన్ వినియోగాన్ని తగ్గిస్తూ ముఖ్య సమాచారం నిలుపుతాయి.
- **Agent scratchpads** ఏ సంభాషణ తగ్గింపు అయినా కొనసాగించే శాశ్వత బాహ్య జ్ఞాపకాన్ని అందిస్తాయి.

### మీరు నిర్మించినవి
- **Context-అవేర్ agent** ఇది బహుళ-తిరుగు సంభాషణలలో కొనసాగించే సామర్థ్యాన్ని కలిగి ఉంటుంది
- **సారాంశ సాధనం** (`summarize_preferences`) ఇది ముఖ్య వినియోగదారు వివరాలను సంక్షిప్త రూపంలో నమోదు చేస్తుంది
- **బహుళ-తిరుగు సంభాషణ** ఇది context నిలుపుకోవడం మరియు మార్పులను నిర్వహించడం చూపిస్తుంది

### వాస్తవ ప్రపంచ అనువర్తనాలు
- **కస్టమర్ సర్వీస్ బాట్లు**: దీర్ఘమైన మద్దతు సત્રలలో ప్రిఫరెన్సులను గుర్తుంచుకోవాలి
- **వ్యక్తిగత సహాయకులు**: పునః వివరణ అవసరం లేకుండా ప్రాజెక్ట్ లను ట్రాక్ చేస్తారు
- **విద్యా ట్యూటర్ల్లు**: అనేక సంభాషణల్లో విద్యార్థి పురోగతిని నిలుపుకుంటారు

### తరువాతి اقدامات
- ఫైల్ ఆధారిత శాశ్వతతతో కూడిన పూర్తి scratchpad సాధనాన్ని అమలు చేయండి
- సారాంశం తర్వాత స్వయంచాలక చరిత్ర తుడవడం జోడించండి
- సాందర్భిక జ్ఞాపక శోధన కోసం వెక్టర్ డేటాబేస్ లతో అనుసంధానం చేయండి
- పూర్తిస్థాయి context తో రోజులు తరువాత సంభాషణలను పునఃప్రారంభించగల ఏజెంట్లు నిర్మించండి


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**విమర్శించు**:  
ఈ డాక్యుమెంట్‌ని AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నించినప్పటికీ, స్వయంచాలక అనువాదాలలో పొరపాటులు లేదా తప్పుడు వివరాలు ఉండవచ్చు అని దయచేసి గమనించండి. అసలైన భాషలో ఉన్న డాక్యుమెంట్‌ను అధికారిక మూలంగా పరిగణించాలి. ముఖ్యమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సూచిస్తాము. ఈ అనువాదం వలన కలిగే ఏవైనా తప్పుకోపాలు లేదా తప్పైన అర్థంప్రాప్తుల కోసం మేము బాధ్యతాయుతులమేము కాదండి.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
